In [38]:
from azure.ai.ml import MLClient, automl, Input
from azure.identity import DefaultAzureCredential
import mlflow
import os

In [67]:
ml_client =ml_client = MLClient.from_config(credential=DefaultAzureCredential())
mlflow_client = mlflow.tracking.MlflowClient()

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [45]:
# Registered Azure ML data asset
training_data =Input(
        type="mltable",
        path="azureml:transaction-train-data:1"
)
validation_data = Input(
        type="mltable",
        path="azureml:transaction-val-data:1"
)
test_data = Input(
        type="mltable",
        path="azureml:transaction-test-data:1"
)
TARGET_COLUMN = "is_fraud"

# Create the AutoML classification job
fraud_detection_job = automl.classification(
    training_data=training_data,
    validation_data=validation_data,
    test_data=test_data,
    target_column_name=TARGET_COLUMN,
    primary_metric="norm_macro_recall",
    experiment_name="Default",
    compute="durrain20031",
    enable_model_explainability=True,

)
fraud_detection_job.set_limits(
    max_trials=6,
    max_concurrent_trials=2,
)

# Submit the job
returned_job = ml_client.jobs.create_or_update(fraud_detection_job)
job_name = returned_job.name
print(f"Submitted job: {job_name}")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Submitted job: gentle_dinner_4jdfgsnt12


In [2]:
job_name = 'gentle_dinner_4jdfgsnt12'

In [63]:
job = ml_client.jobs.get(job_name)
job

Experiment,Name,Type,Status,Details Page
Default,gentle_dinner_4jdfgsnt12,automl,Completed,Link to Azure Machine Learning studio


In [68]:
best_child_run_id = mlflow.get_run(job_name).data.tags['model_explain_best_run_child_id']
best_run = mlflow.get_run(best_child_run_id)

local_path = f"{os.path.dirname(os.getcwd())}/mlflow_artifacts"

# Download run's artifacts/outputs
local_path = mlflow_client.download_artifacts(
    best_run.info.run_id, "outputs", local_path
)
print(f"Artifacts downloaded in: {local_path}")
print(f"Artifacts: {os.listdir(local_path)}")

Artifacts downloaded in: /mnt/batch/tasks/shared/LS_root/mounts/clusters/durrain20031/code/FinRisk360/azureml/mlflow_artifacts/outputs
Artifacts: ['conda_env_v_1_0_0.yml', 'engineered_feature_names.json', 'env_dependencies.json', 'featurization_summary.json', 'generated_code', 'mlflow-model', 'model.pkl', 'pipeline_graph.json', 'run_id.txt', 'scoring_file_pbi_v_1_0_0.py', 'scoring_file_v_1_0_0.py', 'scoring_file_v_2_0_0.py']


In [70]:
import sklearn

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
''